<a href="https://colab.research.google.com/github/megumihoshino/Applied-Machine-Learning-/blob/main/INKA_Project_iseng2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'plotly', 'pandas', '-q'])

# ── Imports ───────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML
import json, os

# ═════════════════════════════════════════════════════════════════
# LOAD DATA
# ═════════════════════════════════════════════════════════════════
NMS_BRANDS = {
    'TOHNICHI','ENERPAC','ELORA','KRISBOW','SHINWA','NFSK','TOX',
    'NIIGATA SEIKI','URYU','HIOKI','FLUKE','KYORITSU','ELCOMETER',
    'SANWA','DAWELL','MILWAUKEE','LEDLENSER','SHINYEI','TOKIN','SKF','IKO',
}

CSV_PATH = '/content/items_clean.csv'
if not os.path.exists(CSV_PATH):
    print("⚠️  items_clean.csv tidak ditemukan. Upload dulu ke Colab!")
else:
    df = pd.read_csv(CSV_PATH)
    df['date_parsed']  = pd.to_datetime(df['date'], format='%m/%d/%Y %H:%M:%S')
    df['month']        = df['date_parsed'].dt.to_period('M').astype(str)
    df['lokasi_clean'] = df['lokasi'].str.replace(r'\s*\(SJ\)', '', regex=True).str.strip()
    df['brand_cat']    = df['brand'].apply(
        lambda b: 'NMS Product' if str(b).upper() in NMS_BRANDS else 'General / Umum'
    )
    df['sell_label']   = df['sell_type'].map({
        'NMS PRODUCT': 'NMS Product (Branded)',
        'GENERAL':     'General / Umum',
    })

    # ── Palette ──
    C = {
        'navy':  '#1B2A4A',
        'blue':  '#2563EB',
        'gold':  '#D97706',
        'green': '#059669',
        'red':   '#DC2626',
        'sky':   '#DBEAFE',
    }
    PAL = ['#2563EB','#D97706','#059669','#DC2626','#7C3AED',
           '#0891B2','#BE185D','#65A30D','#EA580C','#0F766E',
           '#4F46E5','#B45309']

    def fmt_idr(v):
        if v >= 1e9: return f"Rp {v/1e9:.2f}B"
        if v >= 1e6: return f"Rp {v/1e6:.1f}M"
        return f"Rp {v:,.0f}"

    def plotly_cfg():
        return dict(
            plot_bgcolor='white', paper_bgcolor='white',
            font=dict(family='Inter, sans-serif', color='#374151')
        )

    total_rev  = df['item_total'].sum()
    total_so   = df['so_no'].nunique()
    brand_sum  = df.groupby('brand')['item_total'].sum()
    top_brand  = brand_sum.idxmax()
    monthly_g  = df.groupby('month')['item_total'].sum()
    peak_month = monthly_g.idxmax()
    nms_rev    = df[df['sell_type']=='NMS PRODUCT']['item_total'].sum()
    nms_pct    = nms_rev / total_rev * 100

    # ═══════════════════════════════════════════════════════════════
    # BUILD CHARTS → JSON for embedding
    # ═══════════════════════════════════════════════════════════════

    # 1. Area Pie
    area_g = df.groupby('lokasi_clean')['item_total'].sum().reset_index()
    fig_area = px.pie(area_g, names='lokasi_clean', values='item_total',
                      color_discrete_sequence=PAL, hole=0.55)
    fig_area.update_traces(textinfo='label+percent', textposition='outside',
                           pull=[0.04]*len(area_g))
    fig_area.update_layout(**plotly_cfg(), height=300,
                           title=dict(text='Revenue by Area', x=0.5,
                                      font=dict(size=14, color=C['navy'])),
                           showlegend=False, margin=dict(t=50,b=40,l=20,r=20))

    # 2. Sell Type Pie
    sell_g = df.groupby('sell_label')['item_total'].sum().reset_index()
    fig_sell = px.pie(sell_g, names='sell_label', values='item_total',
                      color_discrete_sequence=[C['green'], C['blue']], hole=0.55)
    fig_sell.update_traces(textinfo='label+percent', textposition='outside',
                           pull=[0.04]*len(sell_g))
    fig_sell.update_layout(**plotly_cfg(), height=300,
                           title=dict(text='Revenue by Sell Type', x=0.5,
                                      font=dict(size=14, color=C['navy'])),
                           showlegend=False, margin=dict(t=50,b=40,l=20,r=20))

    # 3. Monthly Bar + Line
    monthly_df = monthly_g.reset_index()
    monthly_df.columns = ['month','revenue']
    monthly_df = monthly_df.sort_values('month')
    monthly_df['rev_b'] = monthly_df['revenue'] / 1e9
    monthly_df['year']  = monthly_df['month'].str[:4]

    fig_monthly = go.Figure()
    fig_monthly.add_trace(go.Bar(
        x=monthly_df['month'], y=monthly_df['rev_b'],
        name='Revenue',
        marker=dict(
            color=monthly_df['year'].map({'2025': C['blue'], '2026': C['gold']}),
            line=dict(color='white', width=1),
            cornerradius=4,
        ),
        text=monthly_df['rev_b'].apply(lambda v: f'{v:.2f}B'),
        textposition='outside', textfont=dict(size=11),
        hovertemplate='<b>%{x}</b><br>Revenue: Rp %{y:.3f}B<extra></extra>',
    ))
    fig_monthly.add_trace(go.Scatter(
        x=monthly_df['month'], y=monthly_df['rev_b'],
        mode='lines+markers', name='Trend',
        line=dict(color=C['red'], width=2, dash='dot'),
        marker=dict(size=7, color=C['red']),
    ))
    fig_monthly.update_layout(**plotly_cfg(), height=320, bargap=0.3,
                              title=dict(text='Monthly Revenue Trend', x=0.5,
                                         font=dict(size=14, color=C['navy'])),
                              xaxis=dict(showgrid=False, linecolor='#E5E7EB', tickangle=-35),
                              yaxis=dict(gridcolor='#F3F4F6', ticksuffix='B'),
                              legend=dict(orientation='h', y=1.1, x=1, xanchor='right'))

    # 4. Top Brand Bar
    top10 = brand_sum.sort_values(ascending=True).tail(10).reset_index()
    top10.columns = ['brand','revenue']
    top10['rev_m'] = top10['revenue'] / 1e6
    top10['cat']   = top10['brand'].apply(
        lambda b: 'NMS Product' if b.upper() in NMS_BRANDS else 'General / Umum')
    fig_brand = px.bar(
        top10, y='brand', x='rev_m', color='cat', orientation='h',
        color_discrete_map={'NMS Product': C['blue'], 'General / Umum': C['gold']},
        text=top10['rev_m'].apply(lambda v: f'{v:.0f}M'),
        labels={'rev_m': 'Revenue (juta IDR)', 'brand': '', 'cat': ''},
    )
    fig_brand.update_traces(textposition='outside', textfont_size=10)
    fig_brand.update_layout(**plotly_cfg(), height=360,
                            title=dict(text='Top 10 Brands by Revenue', x=0.5,
                                       font=dict(size=14, color=C['navy'])),
                            xaxis=dict(gridcolor='#F3F4F6', ticksuffix='M'),
                            yaxis=dict(showgrid=False),
                            legend=dict(orientation='h', y=1.08, x=0),
                            margin=dict(t=50,b=20,l=10,r=90), bargap=0.25)

    # 5. Heatmap
    top12 = brand_sum.sort_values(ascending=False).head(12).index.tolist()
    hm = df[df['brand'].isin(top12)].groupby(['brand','month'])['item_total'].sum()
    hm_pivot = hm.unstack(fill_value=0) / 1e6
    fig_hm = px.imshow(
        hm_pivot,
        color_continuous_scale=[[0,'#EFF6FF'],[0.5,'#3B82F6'],[1,'#1E3A8A']],
        text_auto='.0f', aspect='auto',
        labels=dict(color='Rp Juta'),
    )
    fig_hm.update_traces(textfont_size=9)
    fig_hm.update_layout(**plotly_cfg(), height=380,
                         title=dict(text='Brand × Month Heatmap (Rp Juta)', x=0.5,
                                    font=dict(size=14, color=C['navy'])),
                         xaxis=dict(tickangle=-40),
                         margin=dict(t=50,b=20,l=10,r=10))

    # ── Serialize charts to JSON ──
    def fig_json(fig):
        return fig.to_json()

    j_area    = fig_json(fig_area)
    j_sell    = fig_json(fig_sell)
    j_monthly = fig_json(fig_monthly)
    j_brand   = fig_json(fig_brand)
    j_hm      = fig_json(fig_hm)

    # ── Area table HTML ──
    area_tbl = df.groupby('lokasi_clean').agg(
        Revenue=('item_total','sum'), SO=('so_no','nunique'), Items=('item_total','count')
    ).reset_index().sort_values('Revenue', ascending=False)

    def area_rows_html(tbl):
        rows = ""
        for _, r in tbl.iterrows():
            rows += f"""<tr>
                <td>{r['lokasi_clean']}</td>
                <td style="text-align:right">Rp {r['Revenue']:,.0f}</td>
                <td style="text-align:right">{r['Revenue']/total_rev*100:.1f}%</td>
                <td style="text-align:center">{r['SO']}</td>
                <td style="text-align:center">{r['Items']}</td>
            </tr>"""
        rows += f"""<tr style="background:#DBEAFE;font-weight:700">
                <td>TOTAL</td>
                <td style="text-align:right">Rp {total_rev:,.0f}</td>
                <td style="text-align:right">100%</td>
                <td style="text-align:center">{total_so}</td>
                <td style="text-align:center">{len(df)}</td>
            </tr>"""
        return rows

    # ── Brand table HTML ──
    brand_full = df.groupby(['brand','brand_cat']).agg(
        Revenue=('item_total','sum'), Items=('item_total','count')
    ).reset_index().sort_values('Revenue', ascending=False)

    def brand_rows_html(tbl):
        rows = ""
        for i, r in enumerate(tbl.itertuples(), 1):
            cat_color = '#DBEAFE' if r.brand_cat == 'NMS Product' else '#FEF3C7'
            rows += f"""<tr>
                <td style="text-align:center;color:#6B7280">{i}</td>
                <td><b>{r.brand}</b></td>
                <td style="text-align:right">Rp {r.Revenue:,.0f}</td>
                <td style="text-align:right">{r.Revenue/total_rev*100:.1f}%</td>
                <td><span style="background:{cat_color};padding:2px 8px;border-radius:6px;
                    font-size:11px;font-weight:600">{r.brand_cat}</span></td>
                <td style="text-align:center">{r.Items}</td>
            </tr>"""
        return rows

    # ═══════════════════════════════════════════════════════════════
    # HTML TEMPLATE
    # ═══════════════════════════════════════════════════════════════
    html = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap" rel="stylesheet">
<style>
* {{ box-sizing: border-box; margin: 0; padding: 0; }}
body {{ font-family: 'Inter', sans-serif; background: #F1F5F9; color: #1F2937; }}

.wrapper {{ max-width: 1280px; margin: 0 auto; padding: 24px 20px; }}

/* Header */
.header {{
    background: linear-gradient(135deg, #0F1E3C 0%, #1E3A8A 60%, #2563EB 100%);
    border-radius: 16px; padding: 28px 32px; margin-bottom: 24px;
    display: flex; align-items: center; gap: 18px;
    box-shadow: 0 8px 32px rgba(30,58,138,.3);
}}
.header-icon {{ font-size: 44px; }}
.header-title {{ font-size: 22px; font-weight: 700; color: #fff; letter-spacing: -.02em; }}
.header-sub   {{ font-size: 13px; color: #93C5FD; margin-top: 4px; }}

/* KPI */
.kpi-row {{ display: grid; grid-template-columns: repeat(4,1fr); gap: 16px; margin-bottom: 24px; }}
.kpi-card {{
    background: #fff; border-radius: 14px; padding: 20px 22px;
    border: 1px solid #E5E7EB; position: relative; overflow: hidden;
    box-shadow: 0 2px 12px rgba(0,0,0,.05);
}}
.kpi-card::before {{
    content: ''; position: absolute; top: 0; left: 0; right: 0;
    height: 3px; border-radius: 14px 14px 0 0;
}}
.kpi-card.blue::before  {{ background: #2563EB; }}
.kpi-card.gold::before  {{ background: #D97706; }}
.kpi-card.green::before {{ background: #059669; }}
.kpi-card.red::before   {{ background: #DC2626; }}
.kpi-label {{ font-size: 10.5px; font-weight: 600; text-transform: uppercase;
              letter-spacing: .08em; color: #6B7280; margin-bottom: 8px; }}
.kpi-value {{ font-size: 24px; font-weight: 700; color: #111827; line-height: 1.1; margin-bottom: 4px; }}
.kpi-sub   {{ font-size: 12px; color: #9CA3AF; }}

/* Section */
.section-head {{
    font-size: 15px; font-weight: 700; color: #1E3A8A;
    padding-bottom: 8px; border-bottom: 2px solid #DBEAFE;
    margin-bottom: 16px; margin-top: 28px;
}}

/* Grid layouts */
.grid-2   {{ display: grid; grid-template-columns: 1fr 1fr;     gap: 16px; }}
.grid-3   {{ display: grid; grid-template-columns: 1fr 1fr 1.6fr; gap: 16px; }}

/* Cards */
.card {{
    background: #fff; border-radius: 14px; padding: 16px;
    border: 1px solid #E5E7EB; box-shadow: 0 2px 10px rgba(0,0,0,.04);
}}

/* Tables */
.tbl {{ width: 100%; border-collapse: collapse; font-size: 13px; }}
.tbl thead tr {{ background: #1E3A8A; }}
.tbl thead th {{
    color: #fff; font-size: 11.5px; font-weight: 600;
    padding: 10px 12px; text-align: left; letter-spacing: .02em;
}}
.tbl tbody tr:nth-child(even) {{ background: #F8FAFF; }}
.tbl tbody tr:hover {{ background: #EFF6FF; }}
.tbl td {{ padding: 8px 12px; border-bottom: 1px solid #F3F4F6; }}

/* Insight cards */
.insight {{
    border-left: 4px solid #2563EB; background: #EFF6FF;
    border-radius: 0 10px 10px 0; padding: 12px 14px;
    margin-bottom: 10px; font-size: 13px; color: #1E3A8A; line-height: 1.6;
}}
.insight.warn  {{ border-color: #D97706; background: #FFFBEB; color: #78350F; }}
.insight.good  {{ border-color: #059669; background: #ECFDF5; color: #064E3B; }}

/* Support table badges */
.badge {{ border-radius: 6px; padding: 2px 10px; font-weight: 700; font-size: 11px; }}
.badge-high   {{ background: #FEE2E2; color: #991B1B; }}
.badge-med    {{ background: #FEF3C7; color: #92400E; }}
.badge-low    {{ background: #D1FAE5; color: #065F46; }}

/* Tab navigation */
.tab-bar {{ display: flex; gap: 6px; margin-bottom: 16px; flex-wrap: wrap; }}
.tab-btn {{
    padding: 8px 18px; border-radius: 8px; border: 1.5px solid #DBEAFE;
    background: #fff; cursor: pointer; font-size: 13px; font-weight: 500;
    color: #3B82F6; transition: all .15s;
}}
.tab-btn.active {{ background: #2563EB; color: #fff; border-color: #2563EB; }}
.tab-content {{ display: none; }}
.tab-content.active {{ display: block; }}

/* Footer */
.footer {{
    text-align: center; font-size: 12px; color: #9CA3AF;
    padding: 20px 0; margin-top: 32px;
    border-top: 1px solid #E5E7EB;
}}
</style>
</head>
<body>
<div class="wrapper">

  <!-- Header -->
  <div class="header">
    <div class="header-icon">🚂</div>
    <div>
      <div class="header-title">Laporan Penjualan PT. INKA</div>
      <div class="header-sub">PT. Nasional Makmur Sejahtera &nbsp;·&nbsp; Periode Januari 2025 – 25 Februari 2026</div>
    </div>
  </div>

  <!-- KPI -->
  <div class="kpi-row">
    <div class="kpi-card blue">
      <div class="kpi-label">Total Revenue</div>
      <div class="kpi-value">{fmt_idr(total_rev)}</div>
      <div class="kpi-sub">{total_so} Sales Orders</div>
    </div>
    <div class="kpi-card gold">
      <div class="kpi-label">NMS Product</div>
      <div class="kpi-value">{fmt_idr(nms_rev)}</div>
      <div class="kpi-sub">{nms_pct:.1f}% dari total</div>
    </div>
    <div class="kpi-card green">
      <div class="kpi-label">Top Brand</div>
      <div class="kpi-value">{top_brand}</div>
      <div class="kpi-sub">{fmt_idr(brand_sum.max())}</div>
    </div>
    <div class="kpi-card red">
      <div class="kpi-label">Peak Month</div>
      <div class="kpi-value">{peak_month}</div>
      <div class="kpi-sub">{fmt_idr(monthly_g.max())}</div>
    </div>
  </div>

  <!-- Tabs -->
  <div class="tab-bar">
    <button class="tab-btn active" onclick="showTab('area')">📍 Area & Sell Type</button>
    <button class="tab-btn" onclick="showTab('trend')">📈 Monthly Trend</button>
    <button class="tab-btn" onclick="showTab('brand')">🏷️ Brand Analysis</button>
    <button class="tab-btn" onclick="showTab('heatmap')">🔥 Heatmap</button>
    <button class="tab-btn" onclick="showTab('conclusion')">💡 Conclusion</button>
  </div>

  <!-- TAB: Area & Sell Type -->
  <div id="tab-area" class="tab-content active">
    <div class="section-head">📍 Summary by Area & Product Sell Type</div>
    <div class="grid-3">
      <div class="card"><div id="chart-area"></div></div>
      <div class="card"><div id="chart-sell"></div></div>
      <div class="card" style="overflow:auto;">
        <div style="font-weight:600;font-size:13px;color:#1E3A8A;margin-bottom:12px">Detail per Area</div>
        <table class="tbl">
          <thead><tr><th>Area</th><th>Revenue (IDR)</th><th>Share</th><th>SO</th><th>Items</th></tr></thead>
          <tbody>{area_rows_html(area_tbl)}</tbody>
        </table>
      </div>
    </div>
  </div>

  <!-- TAB: Monthly Trend -->
  <div id="tab-trend" class="tab-content">
    <div class="section-head">📈 Monthly Revenue Trend</div>
    <div class="card"><div id="chart-monthly"></div></div>
  </div>

  <!-- TAB: Brand Analysis -->
  <div id="tab-brand" class="tab-content">
    <div class="section-head">🏷️ Top Brands by Revenue</div>
    <div class="grid-2">
      <div class="card"><div id="chart-brand"></div></div>
      <div class="card" style="overflow:auto;max-height:420px;">
        <div style="font-weight:600;font-size:13px;color:#1E3A8A;margin-bottom:12px">Semua Brand</div>
        <table class="tbl">
          <thead><tr><th>#</th><th>Brand</th><th>Revenue</th><th>%</th><th>Kategori</th><th>Items</th></tr></thead>
          <tbody>{brand_rows_html(brand_full)}</tbody>
        </table>
      </div>
    </div>
  </div>

  <!-- TAB: Heatmap -->
  <div id="tab-heatmap" class="tab-content">
    <div class="section-head">🔥 Brand × Month Heatmap</div>
    <div class="card"><div id="chart-hm"></div></div>
  </div>

  <!-- TAB: Conclusion -->
  <div id="tab-conclusion" class="tab-content">
    <div class="section-head">💡 Conclusion & Support Needed</div>
    <div class="grid-2">
      <div>
        <div style="font-weight:600;font-size:13px;color:#1E3A8A;margin-bottom:12px">📌 Kesimpulan</div>
        <div class="insight">
          <b>Dominasi Madiun:</b> Area Madiun menyumbang 99.6% dari total revenue ({fmt_idr(df[df['lokasi_clean']=='MADIUN']['item_total'].sum())}).
          Banyuwangi masih sangat kecil ({fmt_idr(df[df['lokasi_clean']=='BANYUWANGI']['item_total'].sum())}).
        </div>
        <div class="insight good">
          <b>NMS Product {nms_pct:.1f}% dari revenue</b> ({fmt_idr(nms_rev)}).
          Brand Tohnichi, Enerpac & Elora adalah top kontributor — margin lebih tinggi, perlu didorong lebih agresif.
        </div>
        <div class="insight warn">
          <b>LRT Jakarta belum tercatat</b> dalam periode ini. Area ini merupakan peluang besar yang belum terpenetrasi.
        </div>
        <div class="insight">
          <b>Peak season Q3–Q4 2025</b> (Jun Rp 1.2B, Okt Rp 948M).
          Pola siklus ini harus dijadikan acuan forecast 2026.
        </div>
        <div class="insight good">
          <b>Revenue awal 2026 (Jan–Feb) = {fmt_idr(df[df['month'].str.startswith('2026')]['item_total'].sum())}</b>.
          Wajar untuk awal tahun, pipeline Q2–Q3 2026 perlu segera dibentuk.
        </div>
      </div>
      <div>
        <div style="font-weight:600;font-size:13px;color:#1E3A8A;margin-bottom:12px">🛠️ Support Needed</div>
        <table class="tbl">
          <thead><tr><th>#</th><th>Area</th><th>Action</th><th>Priority</th></tr></thead>
          <tbody>
            <tr><td>1</td><td>LRT Jakarta</td><td>Aktifkan penetrasi — mapping kebutuhan alat proyek LRT</td><td><span class="badge badge-high">HIGH</span></td></tr>
            <tr><td>2</td><td>Banyuwangi</td><td>Tingkatkan frekuensi kunjungan; potensi besar belum tergarap</td><td><span class="badge badge-high">HIGH</span></td></tr>
            <tr><td>3</td><td>NMS Product</td><td>Program bundling Tohnichi + Enerpac; target kontribusi 30%+</td><td><span class="badge badge-med">MEDIUM</span></td></tr>
            <tr><td>4</td><td>Pipeline 2026</td><td>Susun forecast Q2–Q4 berbasis historis; pastikan SO masuk sebelum Q2</td><td><span class="badge badge-med">MEDIUM</span></td></tr>
            <tr><td>5</td><td>Brand Baru</td><td>Introduksi Niigata Seiki & brand premium lain yg belum masuk portofolio</td><td><span class="badge badge-low">LOW</span></td></tr>
          </tbody>
        </table>
      </div>
    </div>
  </div>

  <div class="footer">
    PT. Nasional Makmur Sejahtera &nbsp;·&nbsp; Data: Laporan SO KRESNA &nbsp;·&nbsp; Periode: 01 Jan 2025 – 25 Feb 2026
  </div>

</div>

<script>
// Render all charts
Plotly.newPlot('chart-area',    {json}.parse('{{}}'.replace('{{}}', '')), {{}} , {{responsive:true}});
</script>

<script>
var charts = {{
  'chart-area':    {j_area},
  'chart-sell':    {j_sell},
  'chart-monthly': {j_monthly},
  'chart-brand':   {j_brand},
  'chart-hm':      {j_hm},
}};

for (var id in charts) {{
  var spec = charts[id];
  Plotly.newPlot(id, spec.data, spec.layout, {{responsive: true, displayModeBar: false}});
}}

function showTab(name) {{
  document.querySelectorAll('.tab-content').forEach(el => el.classList.remove('active'));
  document.querySelectorAll('.tab-btn').forEach(el => el.classList.remove('active'));
  document.getElementById('tab-' + name).classList.add('active');
  event.target.classList.add('active');

  // Re-render chart in active tab to fix sizing
  var chartIds = {{
    area: ['chart-area','chart-sell'],
    trend: ['chart-monthly'],
    brand: ['chart-brand'],
    heatmap: ['chart-hm'],
    conclusion: [],
  }};
  (chartIds[name] || []).forEach(id => Plotly.relayout(id, {{autosize: true}}));
}}
</script>
</body>
</html>
"""

    # ── Display in Colab ──
    from IPython.display import display, HTML
    display(HTML(f'<div style="width:100%;height:900px;overflow:auto;border-radius:12px;">{html}</div>'))
    print("✅ Dashboard berhasil ditampilkan!")

✅ Dashboard berhasil ditampilkan!
